In [2]:
from cryptography.hazmat.primitives.asymmetric import rsa
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.backends import default_backend

#generar la clave privada

private_key = rsa.generate_private_key(public_exponent=65537,key_size=2048,backend=default_backend())

#guardar clave

with open("private_key.pem", "wb") as key_file:
  key_file.write(private_key.private_bytes(encoding=serialization.Encoding.PEM, format=serialization.PrivateFormat.PKCS8,encryption_algorithm=serialization.NoEncryption()))



In [5]:
public_key = private_key.public_key()

with open("public_key.pem", "wb") as key_file:
  key_file.write(public_key.public_bytes(encoding=serialization.Encoding.PEM, format=serialization.PublicFormat.SubjectPublicKeyInfo))

In [6]:
# CertificateBuilder permite crear (construir) certificados X.509 paso a paso
from cryptography.x509 import CertificateBuilder

# Conjunto de identificadores (OID) para los distintos campos de un nombre X.509
from cryptography.x509.oid import NameOID

# Algoritmos de hash que se usarán después para firmar el certificado
from cryptography.hazmat.primitives import hashes

# padding define esquemas de relleno para operaciones de cifrado/firma
from cryptography.hazmat.primitives.asymmetric import padding

# Name y NameAttribute representan un “nombre distinguido” (DN) y sus atributos
from cryptography.x509 import Name, NameAttribute

# serialization ofrece utilidades para convertir claves a/desde PEM, DER, etc.
from cryptography.hazmat.primitives import serialization

# default_backend expone la implementación criptográfica por defecto (OpenSSL)
from cryptography.hazmat.backends import default_backend

# random_serial_number genera un número de serie aleatorio y único para el certificado
from cryptography.x509 import random_serial_number

# Módulo estándar para manejar fechas y horas
import datetime



# ---------------------------------------------------------------------------
# CARGAR LA CLAVE PÚBLICA
# ---------------------------------------------------------------------------

with open("public_key.pem", "rb") as key_file:          # Abre el archivo PEM en modo binario
    public_key = serialization.load_pem_public_key(     # Convierte los bytes PEM en objeto PublicKey
        key_file.read(),                                #  -> lee el contenido del fichero
        backend=default_backend()                       #  -> indica qué backend criptográfico usar
    )



# ---------------------------------------------------------------------------
# CREAR EL CERTIFICADO (estructura todavía sin firmar)
# ---------------------------------------------------------------------------

builder = CertificateBuilder(                           # Instancia el “builder”
    subject_name=Name([                                 # 1‑‑ Nombre del sujeto (propietario)
        NameAttribute(NameOID.COUNTRY_NAME, u"ES"),
        NameAttribute(NameOID.STATE_OR_PROVINCE_NAME, u"Madrid"),
        NameAttribute(NameOID.LOCALITY_NAME, u"Madrid"),
        NameAttribute(NameOID.ORGANIZATION_NAME, u"MiEmpresa"),
        NameAttribute(NameOID.COMMON_NAME, u"www.miempresa.com")
    ]),
    issuer_name=Name([                                  # 2‑‑ Nombre del emisor (CA).
        NameAttribute(NameOID.COUNTRY_NAME, u"ES"),     #     Aquí se pone igual que el sujeto
        NameAttribute(NameOID.STATE_OR_PROVINCE_NAME, u"Madrid"),   #     porque es un certificado
        NameAttribute(NameOID.LOCALITY_NAME, u"Madrid"),            #     autofirmado (self‑signed).
        NameAttribute(NameOID.ORGANIZATION_NAME, u"MiEmpresa"),
        NameAttribute(NameOID.COMMON_NAME, u"www.miempresa.com")
    ]),
    not_valid_before=datetime.datetime.utcnow(),        # 3‑‑ Fecha/hora de inicio de validez (ahora)
    not_valid_after=datetime.datetime.utcnow() +        # 4‑‑ Fecha/hora de caducidad (1 año)
                     datetime.timedelta(days=365),
    public_key=public_key,                              # 5‑‑ Clave pública que se certifica
)



# ---------------------------------------------------------------------------
# CARGAR LA CLAVE PRIVADA QUE FIRMARÁ EL CERTIFICADO
# ---------------------------------------------------------------------------

with open("private_key.pem", "rb") as key_file:         # Abre la clave privada en modo binario
    private_key = serialization.load_pem_private_key(   # Crea objeto PrivateKey desde PEM
        key_file.read(),                                #  -> contenido del fichero
        password=None,                                  #  -> contraseña si estuviera cifrada
        backend=default_backend()
    )



# ---------------------------------------------------------------------------
# AÑADIR NÚMERO DE SERIE Y FIRMAR
# ---------------------------------------------------------------------------

builder = builder.serial_number(                        # Asigna un número de serie único
    random_serial_number()
)

certificate = builder.sign(                             # Firma el certificado (lo hace “oficial”)
    private_key=private_key,                            #  -> con la clave privada cargada
    algorithm=hashes.SHA256(),                          #  -> usando SHA‑256 como hash
    backend=default_backend()
)



# ---------------------------------------------------------------------------
# GUARDAR EL CERTIFICADO FIRMADO EN FORMATO PEM
# ---------------------------------------------------------------------------

with open("certificate.pem", "wb") as cert_file:        # Crea/abre el fichero destino en binario
    cert_file.write(                                    # Escribe los bytes del certificado
        certificate.public_bytes(serialization.Encoding.PEM)  #  -> codificados en PEM (Base‑64)
    )


/tmp/ipykernel_11017/2741863437.py:61: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  not_valid_before=datetime.datetime.utcnow(),        # 3‑‑ Fecha/hora de inicio de validez (ahora)
/tmp/ipykernel_11017/2741863437.py:62: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  not_valid_after=datetime.datetime.utcnow() +        # 4‑‑ Fecha/hora de caducidad (1 año)


In [7]:
!openssl pkcs12 -export -out certificado.p12 -inkey private_key.pem -in certificate.pem -name "MiEmpresa firma digital" -password pass:cambiar_por_una_segura

# si pongo vacio -password pass:  me pedirá una contraseña